# Example 10.4.3 / 2: Functions the MP Driver Approximates

Companion notebook for **10.4.3 Automatic Piecewise-Linear Approximation** in the AMPL lab Documentation, section "Functions the MP Driver Approximates". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Functions the MP Driver Approximates

For linear-quadratic MP-based solvers, MP's documentation states that most supported nonlinear
functions are handled by piecewise-linear approximation, except products with binary variables.
Each approximated expression produces a driver warning named `PLApprox_<Type>`. The run below
defines one auxiliary value per function over a bounded argument `X` and lists the warning types
that HiGHS reported:

| Function | Example in the model | Warning type reported by HiGHS |
|---|---|---|
| `log` | `log(1 + X)` | `PLApprox_Log` |
| `log10` | `log10(1 + X)` | `PLApprox_LogA` |
| `exp` | `exp(X)` | `PLApprox_Exp` |
| `sin` | `sin(X)` | `PLApprox_Sin` |
| `cos` | `cos(X)` | `PLApprox_Cos` |
| `tan` | `tan(X)` | `PLApprox_Tan` |
| `asin` | `asin(X)` | `PLApprox_Asin` |
| `acos` | `acos(X)` | `PLApprox_Acos` |
| `atan` | `atan(X)` | `PLApprox_Atan` |
| `sinh` | `sinh(X)` | `PLApprox_Sinh` |
| `cosh` | `cosh(X)` | `PLApprox_Cosh` |
| `tanh` | `tanh(X)` | `PLApprox_Tanh` |
| `asinh` | `asinh(X)` | `PLApprox_Asinh` |
| `acosh` | `acosh(1 + X)` | `PLApprox_Acosh` |
| `atanh` | `atanh(X)` | `PLApprox_Atanh` |
| `^` with a constant exponent | `X^1.5`, and `sqrt(X)` | `PLApprox_PowConstExp` |
| `^` with a constant positive base | `2^X` | `PLApprox_ExpA` |

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var X >= 0.1, <= 0.9;  # argument kept inside every function's domain
var Value {1..17};  # value of each nonlinear function

minimize Smallest: X;  # simple objective; the constraints are the point

subject to Log:   Value[1]  = log(1 + X);  # reported as PLApprox_Log
subject to Log10: Value[2]  = log10(1 + X);  # reported as PLApprox_Log10
subject to Exp:   Value[3]  = exp(X);  # reported as PLApprox_Exp
subject to Sin:   Value[4]  = sin(X);  # reported as PLApprox_Sin
subject to Cos:   Value[5]  = cos(X);  # reported as PLApprox_Cos
subject to Tan:   Value[6]  = tan(X);  # reported as PLApprox_Tan
subject to Asin:  Value[7]  = asin(X);  # reported as PLApprox_Asin
subject to Acos:  Value[8]  = acos(X);  # reported as PLApprox_Acos
subject to Atan:  Value[9]  = atan(X);  # reported as PLApprox_Atan
subject to Sinh:  Value[10] = sinh(X);  # reported as PLApprox_Sinh
subject to Cosh:  Value[11] = cosh(X);  # reported as PLApprox_Cosh
subject to Tanh:  Value[12] = tanh(X);  # reported as PLApprox_Tanh
subject to Asinh: Value[13] = asinh(X);  # reported as PLApprox_Asinh
subject to Acosh: Value[14] = acosh(1 + X);  # reported as PLApprox_Acosh
subject to Atanh: Value[15] = atanh(X);  # reported as PLApprox_Atanh
subject to Power: Value[16] = X^1.5 + sqrt(X);  # both give PLApprox_PowConstExp
subject to ExpA:  Value[17] = 2^X;  # constant base

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
import re

# return_output=True returns the solver's messages instead of printing them
log = ampl.solve(solver="highs", verbose=False, return_output=True)
print(ampl.solve_result)
print(sorted(set(re.findall(r'"(PLApprox_\w+)"', log))))

solved
['PLApprox_Acos', 'PLApprox_Acosh', 'PLApprox_Asin', 'PLApprox_Asinh', 'PLApprox_Atan', 'PLApprox_Atanh', 'PLApprox_Cos', 'PLApprox_Cosh', 'PLApprox_Exp', 'PLApprox_ExpA', 'PLApprox_Log', 'PLApprox_LogA', 'PLApprox_PowConstExp', 'PLApprox_Sin', 'PLApprox_Sinh', 'PLApprox_Tan', 'PLApprox_Tanh']
